In [18]:
import numpy as np
import pandas as pd

In [19]:
ENTRY_AGE = 35
POLICY_TERM = 20
SUM_ASSURED = 1_000_000
INTEREST_RATE = 0.05

v = 1 / (1 + INTEREST_RATE)

INITIAL_EXPENSE = 1000
RENEWAL_EXPENSE = 200
COMMISSION_RATE = 0.10
PREMIUM_COLLECTION_EXPENSE = 50

In [20]:
# 2. Load mortality dataset
mortality_data = pd.read_excel(
    "D:\Actuary\Projects Actuarial_Practice\Life_insurance_engine\data\mortality_tables\IIAM 12-15.xlsx",
)
mortality_data.head()

,Age,qx
0,20,0.000284
1,21,0.000305
2,22,0.000328
3,23,0.000353
4,24,0.000379


In [21]:
def mortality_table(ENTRY_AGE, POLICY_TERM, mortality_data):

    # Select mortality rates from dataset
    df = mortality_data[
        (mortality_data["Age"] >= ENTRY_AGE) &
        (mortality_data["Age"] < ENTRY_AGE + POLICY_TERM)
    ].copy()

    df = df.sort_values("Age").reset_index(drop=True)

    if len(df) != POLICY_TERM:
        raise ValueError("Mortality data is incomplete for the selected ages.")

    # Policy year
    df["Policy Year"] = range(1, POLICY_TERM + 1)

    # Individual probability of death
    df["Individual Probability of Death"] = df["qx"]

    # Initialize cohort
    lives_start = []
    deaths = []
    lives_end = []

    lives = 1.0

    for qx in df["qx"]:

        lives_start.append(lives)

        dx = lives * qx
        deaths.append(dx)

        lives = lives - dx
        lives_end.append(lives)

    # Add calculated columns
    df["Number of Lives at Start"] = lives_start
    df["Number of Deaths"] = deaths
    df["Number of Lives at End"] = lives_end

    return df[
        [
            "Policy Year",
            "Age",
            "Individual Probability of Death",
            "Number of Lives at Start",
            "Number of Deaths",
            "Number of Lives at End"
        ]
    ]

In [23]:
mortality = mortality_table(
    ENTRY_AGE=ENTRY_AGE,
    POLICY_TERM=POLICY_TERM,
    mortality_data=mortality_data
)

display(mortality.round(6))

,Policy Year,Age,Individual Probability of Death,Number of Lives at Start,Number of Deaths,Number of Lives at End
0,1,35,0.000847,1.000000,0.000847,0.999153
1,2,36,0.000913,0.999153,0.000912,0.998241
2,3,37,0.000984,0.998241,0.000982,0.997259
3,4,38,0.001061,0.997259,0.001058,0.996200
4,5,39,0.001144,0.996200,0.001140,0.995061
5,6,40,0.001234,0.995061,0.001228,0.993833
6,7,41,0.001332,0.993833,0.001324,0.992509
7,8,42,0.001438,0.992509,0.001427,0.991082
8,9,43,0.001553,0.991082,0.001539,0.989543
9,10,44,0.001679,0.989543,0.001661,0.987881
